In [1]:
import numpy as np
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

from helpers import *
from implementations import *

In [2]:
import os
import zipfile

DATA_PATH = "data/"

zip_path = os.path.join(DATA_PATH, "dataset_to_release.zip")

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(DATA_PATH)

print(os.listdir(DATA_PATH))

['dataset', 'dataset_to_release.zip']


In [6]:
"""
Baseline: computes the reference F1 that everyone tries to beat.
"""

import numpy as np
from preprocessing import *
from helpers import load_csv_data

# ---------------------------------------------------------------------
# Load the data
# Replace this part with the loading code you already use, so that you
# get: x_train, x_test, y_train (labels -1/1) and names (column names).
# ---------------------------------------------------------------------
x_train, x_test, y_train, train_ids, test_ids = load_csv_data("data/dataset")

import csv
import numpy as np

with open("data/dataset/x_train.csv", "r") as f: #ouvrir un fichier csv, le lire et récupérer l'entête
    header = next(csv.reader(f)) #recupere seule l'entête du fichier csv
names = header[1:]   # on enlève "Id" car on ne veut pas l'analyser
assert len(names) == x_train.shape[1]

rows = []
for j, name in enumerate(names):
    col = x_train[:, j]
    valid = col[~np.isnan(col)]
    n_unique = len(np.unique(valid))
    rows.append([
        j, name,
        round(np.isnan(col).mean(), 3),          # la part de cases vides. Python compte True comme 1 et False comme 0, 
        #donc la moyenne donne directement le pourcentage de vides. round(..., 3) arrondit à 3 chiffres après la virgule.
        n_unique,                                 # nombre de valeurs distinctes
        valid.min() if len(valid) else "",
        valid.max() if len(valid) else "",
        round(valid.mean(), 3) if len(valid) else "",#si la colonne a au moins une réponse, calcule ; sinon, laisse la case vide
        ])
    
y01 = np.where(y_train == 1, 1, 0)  # -1/1 labels -> 0/1
# ---------------------------------------------------------------------
# Train without cleaning, evaluate on 20 % of the training set
# ---------------------------------------------------------------------
print("Baseline: train without cleaning, evaluate on 20 % of the training set")
keep_raw = fit_column_filter(x_train)
x_train_raw = x_train[:, keep_raw]
print("Raw columns kept:", x_train_raw.shape[1])
evaluate(x_train_raw, y01)

# ---------------------------------------------------------------------
# Cleaning (rules learned on x_train, applied identically to x_test)
# ---------------------------------------------------------------------
groups = compute_groups(x_train, names)

x_train_clean, kept_names = clean(x_train, names, groups)
x_test_clean, _ = clean(x_test, names, groups)

keep = fit_column_filter(x_train_clean)
x_train_clean = x_train_clean[:, keep]
x_test_clean = x_test_clean[:, keep]
kept_names = [n for n, k in zip(kept_names, keep) if k]
print("Columns kept:", len(kept_names))



# ---------------------------------------------------------------------
# Train on 80 %, evaluate on 20 %
# ---------------------------------------------------------------------
evaluate(x_train_clean, y01)


Baseline: train without cleaning, evaluate on 20 % of the training set
Raw columns kept: 218
F1 (validation) = 0.409  with threshold 0.22  (218 columns)
Columns kept: 170
F1 (validation) = 0.422  with threshold 0.20  (170 columns)


np.float64(0.42179377508718824)

In [9]:
# --- Step 2: one-hot encoding of categorical columns
from preprocessing import *
CATEGORICAL = ["MARITAL", "EMPLOY1", "RENTHOM1", "PERSDOC2",
               "_RACE", "SXORIENT", "TRNSGNDR"]

x_train_oh, x_test_oh, names_oh = one_hot(x_train_clean, x_test_clean, kept_names, CATEGORICAL)

# checks
print("Columns:", x_train_clean.shape[1], "->", x_train_oh.shape[1])
print("Same number of columns in train and test:", x_train_oh.shape[1] == x_test_oh.shape[1])
print("Names match columns:", len(names_oh) == x_train_oh.shape[1])

cols = [i for i, n in enumerate(names_oh) if n.startswith("MARITAL=")]
print("MARITAL sums per row:", np.unique(x_train_oh[:, cols].sum(axis=1)))   # should be [0. 1.]

# score
print("One-hot:")
score_oh = evaluate(x_train_oh, y01)

Columns: 170 -> 199
Same number of columns in train and test: True
Names match columns: True
MARITAL sums per row: [0. 1.]
One-hot:
F1 (validation) = 0.425  with threshold 0.22  (199 columns)


In [4]:
# ---------------------------------------------------------------------
# Retrain on ALL training data and create the submission file
# ---------------------------------------------------------------------
mean, std = fit_preprocessing(x_train_clean)
x_all = apply_preprocessing(x_train_clean, mean, std)
x_te = apply_preprocessing(x_test_clean, mean, std)

w = train_logistic(x_all, y01)

probs_test = sigmoid(x_te @ w)
y_test_pred = np.where(probs_test >= t, 1, -1)   # same threshold as validation, labels back to -1/1

print("Predicted positives in test:", np.mean(y_test_pred == 1))
create_csv_submission(test_ids, y_test_pred, "submission1.csv")
print("File created: submission1.csv")

iter 0: loss = 0.6670
iter 100: loss = 0.2665
iter 200: loss = 0.2376
iter 300: loss = 0.2298
iter 400: loss = 0.2265
Predicted positives in test: 0.14443357500068568
File created: submission1.csv
